In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np

mpl.rcParams.update(
    {
        "text.usetex": False,
        "axes.labelsize": 20,
        "figure.labelsize": 18,
        "xtick.labelsize": 16,
        "ytick.labelsize": 16,
        "figure.constrained_layout.wspace": 0,
        "figure.constrained_layout.hspace": 0,
        "figure.constrained_layout.h_pad": 0,
        "figure.constrained_layout.w_pad": 0,
        "axes.linewidth": 1.2,
    }
)

import jax
import jax.numpy as jnp

jax.config.update("jax_enable_x64", True)

## Kernel Approximations for Convolved Transfer Functions

This notebook collects the approximation experiments that are separate from the main transfer-function tutorial in `04_TransferFunctions.ipynb`.

The goal is to approximate dense convolved kernels with scalable quasiseparable surrogates using two families:

1. `ExponentialSeries`: a nonnegative mixture of exponentials
2. `SHOSeries`: a nonnegative mixture of overdamped SHO atoms

Both are fit directly in the time domain over the lag range of interest.


In [ ]:
import equinox as eqx
from scipy.optimize import nnls
from tinygp.kernels import Kernel

from eztaox.kernels.eqx_utils import find_param_by_name
from eztaox.kernels.quasisep import Exp, Quasisep, SHO
from eztaox.kernels.transfer_function import (
    ConvolvedKernel,
    CausalGaussianTransferFunction,
    TransferFunction,
)

In [ ]:
class CausalTopHatTransferFunction(TransferFunction):
    """A normalized causal top-hat transfer function."""

    def evaluate(self, X1, X2):
        dt = X2 - X1 - self.shift
        half_width = self.width / 2.0
        inside = (dt >= -half_width) & (dt <= half_width)
        return jnp.where(inside, 1.0 / jnp.maximum(self.width, 1e-12), 0.0)

### 1. Dense convolved kernels

We will compare approximations for two transfer functions applied to the same DRW/OU base kernel:

- a causal Gaussian
- a causal top-hat


In [ ]:
base_kernel = Exp(scale=80.0, sigma=0.2)
gaussian_tf = CausalGaussianTransferFunction(width=80.0, shift=30.0)
tophat_tf = CausalTopHatTransferFunction(width=80.0, shift=30.0)

convolved_kernel = ConvolvedKernel(
    base_kernel=base_kernel,
    transfer_function=gaussian_tf,
    n_grid=2048,
)
convolved_kernel_tophat = ConvolvedKernel(
    base_kernel=base_kernel,
    transfer_function=tophat_tf,
    n_grid=2048,
)

tau_grid = jnp.linspace(0.0, 300.0, 600)
k_base = jax.vmap(lambda tau: base_kernel.evaluate(0.0, tau))(tau_grid)
k_conv = jax.vmap(lambda tau: convolved_kernel.evaluate(0.0, tau))(tau_grid)
k_conv_tophat = jax.vmap(lambda tau: convolved_kernel_tophat.evaluate(0.0, tau))(
    tau_grid
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)
axes[0].plot(np.asarray(tau_grid), np.asarray(k_base), label="Base Exp", lw=2)
axes[0].plot(np.asarray(tau_grid), np.asarray(k_conv), label="Gaussian direct", lw=2)
axes[0].plot(
    np.asarray(tau_grid), np.asarray(k_conv_tophat), label="Top-hat direct", lw=2
)
axes[0].set_xlabel(r"Lag $|\Delta t|$")
axes[0].set_ylabel(r"$k(\Delta t)$")
axes[0].grid(alpha=0.25)
axes[0].legend(frameon=False)

axes[1].plot(np.asarray(tau_grid), np.asarray(k_conv), label="Gaussian direct", lw=2)
axes[1].plot(
    np.asarray(tau_grid), np.asarray(k_conv_tophat), label="Top-hat direct", lw=2
)
axes[1].set_xlabel(r"Lag $|\Delta t|$")
axes[1].set_ylabel(r"$k(\Delta t)$")
axes[1].set_title("Dense target kernels")
axes[1].grid(alpha=0.25)
axes[1].legend(frameon=False)

In [ ]:
class ExponentialSeries(Quasisep):
    """Approximate a stationary kernel as a nonnegative sum of exponentials."""

    scales: jax.Array
    weights: jax.Array

    @classmethod
    def from_kernel(
        cls,
        kernel: Kernel,
        n_terms: int = 24,
        n_fit: int = 1024,
        tau_max: float | None = None,
        scale_min_ratio: float = 0.03,
        scale_max_ratio: float = 30.0,
        weight_floor_ratio: float = 1e-3,
        zero_lag_boost: float = 20.0,
    ) -> "ExponentialSeries":
        kernel_scales = find_param_by_name(kernel, "scale")
        if kernel_scales is None:
            raise ValueError(
                "Kernel must have a 'scale' parameter for auto scale placement."
            )

        base_scale = float(sum(kernel_scales) / len(kernel_scales))
        if tau_max is None:
            tau_max = scale_max_ratio * base_scale

        tau_min = max(scale_min_ratio * base_scale * 1e-2, 1e-6)
        tau_fit = np.concatenate([[0.0], np.geomspace(tau_min, tau_max, n_fit - 1)])
        scales = np.geomspace(
            max(scale_min_ratio * base_scale, 1e-6),
            max(scale_max_ratio * base_scale, scale_min_ratio * base_scale * 1.01),
            n_terms,
        )

        tau_fit_jax = jnp.asarray(tau_fit)
        k_fit = np.asarray(
            jax.vmap(lambda tau: kernel.evaluate(jnp.array(0.0), tau))(tau_fit_jax)
        )
        A = np.exp(-tau_fit[:, None] / scales[None, :])

        k0 = float(k_fit[0])
        sigma_w = np.maximum(np.abs(k_fit), weight_floor_ratio * max(k0, 1e-12))
        sigma_w[0] /= zero_lag_boost
        Aw = A / sigma_w[:, None]
        bw = k_fit / sigma_w
        weights, _ = nnls(Aw, bw)

        wsum = weights.sum()
        if wsum > 0:
            weights *= k0 / wsum

        return cls(scales=jnp.asarray(scales), weights=jnp.asarray(weights))

    def coord_to_sortable(self, X):
        return X

    def design_matrix(self):
        return -jnp.diag(1.0 / self.scales)

    def stationary_covariance(self):
        return jnp.diag(self.weights)

    def observation_model(self, X):
        del X
        return jnp.ones_like(self.scales)

    def transition_matrix(self, X1, X2):
        dt = X2 - X1
        return jnp.diag(jnp.exp(-dt / self.scales))

    def power(self, f, df=None):
        del df
        a0 = 1.0 / self.scales
        num = 2.0 * self.weights * a0
        den = a0**2 + (2.0 * jnp.pi * f) ** 2
        return jnp.sum(num / den, axis=0)

In [ ]:
def stable_sho_cov(tau, omega, q, sigma=1.0):
    tau = jnp.asarray(tau)
    if q < 0.5:
        f = jnp.sqrt(jnp.maximum(1.0 - 4.0 * q**2, 1e-12))
        lam_slow = omega * (1.0 - f) / (2.0 * q)
        lam_fast = omega * (1.0 + f) / (2.0 * q)
        w_slow = 0.5 * (1.0 + 1.0 / f)
        w_fast = 0.5 * (1.0 - 1.0 / f)
        return sigma**2 * (
            w_slow * jnp.exp(-lam_slow * tau) + w_fast * jnp.exp(-lam_fast * tau)
        )
    atom = SHO(omega=omega, quality=q, sigma=sigma)
    return atom.evaluate(0.0, tau)


class SHOSeries(Quasisep):
    """Nonnegative mixture of fixed SHO atoms fit in the time domain."""

    omegas: jax.Array
    qualities: jax.Array
    weights: jax.Array

    @classmethod
    def from_kernel(
        cls,
        kernel,
        n_omega: int = 10,
        q_grid=(0.15, 0.25, 0.4),
        n_fit: int = 512,
        tau_max: float | None = None,
        scale_min_ratio: float = 0.03,
        scale_max_ratio: float = 30.0,
        weight_floor_ratio: float = 1e-3,
        zero_lag_boost: float = 20.0,
    ):
        kernel_scales = find_param_by_name(kernel, "scale")
        if kernel_scales is None:
            raise ValueError("Kernel must have a 'scale' parameter for auto placement.")

        base_scale = float(sum(kernel_scales) / len(kernel_scales))
        if tau_max is None:
            tau_max = scale_max_ratio * base_scale

        tau_min = max(scale_min_ratio * base_scale * 1e-2, 1e-6)
        tau_fit = np.concatenate([[0.0], np.geomspace(tau_min, tau_max, n_fit - 1)])
        tau_basis = np.geomspace(
            max(scale_min_ratio * base_scale, 1e-6),
            max(scale_max_ratio * base_scale, scale_min_ratio * base_scale * 1.01),
            n_omega,
        )
        omegas = 1.0 / tau_basis
        qualities = np.asarray(q_grid, dtype=float)

        tau_fit_jax = jnp.asarray(tau_fit)
        k_fit = np.asarray(
            jax.vmap(lambda tau: kernel.evaluate(jnp.array(0.0), tau))(tau_fit_jax)
        )

        atoms = [(omega, q) for q in qualities for omega in omegas]
        A = np.column_stack(
            [
                np.asarray(stable_sho_cov(tau_fit_jax, omega, q, sigma=1.0))
                for omega, q in atoms
            ]
        )

        k0 = float(k_fit[0])
        sigma_w = np.maximum(np.abs(k_fit), weight_floor_ratio * max(k0, 1e-12))
        sigma_w[0] /= zero_lag_boost
        Aw = A / sigma_w[:, None]
        bw = k_fit / sigma_w
        weights, _ = nnls(Aw, bw)

        wsum = weights.sum()
        if wsum > 0:
            weights *= k0 / wsum

        omega_arr = np.asarray([omega for omega, _ in atoms], dtype=float)
        q_arr = np.asarray([q for _, q in atoms], dtype=float)
        return cls(
            omegas=jnp.asarray(omega_arr),
            qualities=jnp.asarray(q_arr),
            weights=jnp.asarray(weights),
        )

    def _atoms(self):
        return [
            SHO(omega=self.omegas[i], quality=self.qualities[i], sigma=1.0)
            for i in range(self.omegas.shape[0])
        ]

    def coord_to_sortable(self, X):
        return X

    def design_matrix(self):
        atoms = self._atoms()
        return jax.scipy.linalg.block_diag(*[atom.design_matrix() for atom in atoms])

    def stationary_covariance(self):
        atoms = self._atoms()
        return jax.scipy.linalg.block_diag(
            *[
                self.weights[i] * atom.stationary_covariance()
                for i, atom in enumerate(atoms)
            ]
        )

    def observation_model(self, X):
        atoms = self._atoms()
        return jnp.concatenate([atom.observation_model(X) for atom in atoms])

    def transition_matrix(self, X1, X2):
        atoms = self._atoms()
        return jax.scipy.linalg.block_diag(
            *[atom.transition_matrix(X1, X2) for atom in atoms]
        )

    def power(self, f, df=None):
        del df
        out = 0.0
        for i, atom in enumerate(self._atoms()):
            out = out + self.weights[i] * atom.power(f)
        return out

### 2. Gaussian transfer function: compare approximations

We fit both approximations only on the lag range shown in the plot. That avoids spending fit capacity on very long lags that are irrelevant for this comparison and also avoids numerical overflow in the overdamped SHO basis.


In [ ]:
exp_series = ExponentialSeries.from_kernel(
    convolved_kernel,
    n_terms=24,
    n_fit=1024,
    tau_max=float(tau_grid.max()),
    scale_min_ratio=0.03,
    scale_max_ratio=30.0,
)
sho_series = SHOSeries.from_kernel(
    convolved_kernel,
    n_omega=18,
    q_grid=(0.08, 0.12, 0.18, 0.25, 0.35, 0.49),
    n_fit=1024,
    tau_max=250.0,
    zero_lag_boost=50.0,
)


k_exp_series = jax.vmap(lambda tau: exp_series.evaluate(0.0, tau))(tau_grid)
k_sho = jax.vmap(lambda tau: sho_series.evaluate(0.0, tau))(tau_grid)

In [ ]:
frac_err_exp = (k_exp_series - k_conv) / jnp.maximum(jnp.abs(k_conv), 1e-12)
frac_err_sho = (k_sho - k_conv) / jnp.maximum(jnp.abs(k_conv), 1e-12)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)
axes[0].plot(np.asarray(tau_grid), np.asarray(k_conv), label="Gaussian direct", lw=2)
axes[0].plot(
    np.asarray(tau_grid),
    np.asarray(k_exp_series),
    label="Gaussian ExponentialSeries",
    lw=2,
    ls="--",
)
axes[0].plot(
    np.asarray(tau_grid), np.asarray(k_sho), label="Gaussian SHOSeries", lw=2, ls=":"
)
axes[0].set_xlabel(r"Lag $|\Delta t|$")
axes[0].set_ylabel(r"$k(\Delta t)$")
axes[0].set_title("Gaussian kernel comparison")
axes[0].grid(alpha=0.25)
axes[0].legend(frameon=False)

axes[1].plot(
    np.asarray(tau_grid), np.asarray(frac_err_exp), label="ExponentialSeries", lw=2
)
axes[1].plot(np.asarray(tau_grid), np.asarray(frac_err_sho), label="SHOSeries", lw=2)
axes[1].axhline(0.0, color="0.2", lw=1)
axes[1].set_xlabel(r"Lag $|\Delta t|$")
axes[1].set_ylabel("Fractional error")
axes[1].set_title("Gaussian approximation error")
axes[1].grid(alpha=0.25)
axes[1].legend(frameon=False)

### 3. Top-hat transfer function: compare approximations

The top-hat case is harder because of the sharper structure introduced by the transfer function. The same workflow applies.


In [ ]:
exp_series_tophat = ExponentialSeries.from_kernel(
    convolved_kernel_tophat,
    n_terms=24,
    n_fit=1024,
    tau_max=float(tau_grid.max()),
    scale_min_ratio=0.03,
    scale_max_ratio=30.0,
)
sho_series_tophat = SHOSeries.from_kernel(
    convolved_kernel_tophat,
    n_omega=10,
    q_grid=(0.15, 0.25, 0.4),
    n_fit=512,
    tau_max=float(tau_grid.max()),
)

k_exp_series_tophat = jax.vmap(lambda tau: exp_series_tophat.evaluate(0.0, tau))(
    tau_grid
)
k_sho_tophat = jax.vmap(lambda tau: sho_series_tophat.evaluate(0.0, tau))(tau_grid)

In [ ]:
frac_err_exp_tophat = (k_exp_series_tophat - k_conv_tophat) / jnp.maximum(
    jnp.abs(k_conv_tophat), 1e-12
)
frac_err_sho_tophat = (k_sho_tophat - k_conv_tophat) / jnp.maximum(
    jnp.abs(k_conv_tophat), 1e-12
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)
axes[0].plot(
    np.asarray(tau_grid), np.asarray(k_conv_tophat), label="Top-hat direct", lw=2
)
axes[0].plot(
    np.asarray(tau_grid),
    np.asarray(k_exp_series_tophat),
    label="Top-hat ExponentialSeries",
    lw=2,
    ls="--",
)
axes[0].plot(
    np.asarray(tau_grid),
    np.asarray(k_sho_tophat),
    label="Top-hat SHOSeries",
    lw=2,
    ls=":",
)
axes[0].set_xlabel(r"Lag $|\Delta t|$")
axes[0].set_ylabel(r"$k(\Delta t)$")
axes[0].set_title("Top-hat kernel comparison")
axes[0].grid(alpha=0.25)
axes[0].legend(frameon=False)

axes[1].plot(
    np.asarray(tau_grid),
    np.asarray(frac_err_exp_tophat),
    label="ExponentialSeries",
    lw=2,
)
axes[1].plot(
    np.asarray(tau_grid), np.asarray(frac_err_sho_tophat), label="SHOSeries", lw=2
)
axes[1].axhline(0.0, color="0.2", lw=1)
axes[1].set_xlabel(r"Lag $|\Delta t|$")
axes[1].set_ylabel("Fractional error")
axes[1].set_title("Top-hat approximation error")
axes[1].grid(alpha=0.25)
axes[1].legend(frameon=False)

### Notes

- The exponential mixture is simple and cheap, but it struggles with kernels that have a very flat zero-lag shoulder.
- The SHO mixture is more flexible near zero lag, but it is also a richer basis and can require more atoms.
- For the SHO basis, we fit only over the lag range of interest and use a stable overdamped covariance formula to avoid overflow in the notebook construction step.
